# Phase 2 — Train RQ-VAE trên Kaggle

Notebook này chạy source trong thư mục `src/` mà không yêu cầu chuyển project thành Python package. Nó tự tìm source, copy source vào `/kaggle/working`, tạo Gin config theo đường dẫn thực tế, rồi chạy `train_rqvae.py` với working directory đúng.

Đầu vào bắt buộc từ notebook 02:

- `global_product_embeddings.f16.npy`
- `global_embedding_index.parquet`

Đầu ra chính:

- các checkpoint RQ-VAE;
- `semantic_ids.parquet` với ba cột `sid_0`, `sid_1`, `sid_2`;
- `semantic_id_metrics.json`.

Codebook được cố định ở `[128, 64, 32]`; không thêm collision suffix.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Data chứa output của notebook 02.
3. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
4. Tạo Kaggle Secret `WANDB_API_KEY` nếu bật W&B.
5. Nếu tự động tìm sai embedding, điền trực tiếp `EMBEDDING_ROOT` ở cell cấu hình.

## 0. Cấu hình

In [ ]:
from pathlib import Path

SOURCE_ROOT = None
EMBEDDING_ROOT = None
OUTPUT_ROOT = None
PRETRAINED_CHECKPOINT = None

CLONE_SOURCE_FROM_GITHUB = True
GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"
REPOSITORY_ROOT = None

CODEBOOK_SIZES = (128, 64, 32)
ITERATIONS = 50_000
BATCH_SIZE = 1_024
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EVAL_FRACTION = 0.05
EVAL_EVERY = 2_500
SAVE_MODEL_EVERY = 5_000
SEED = 2026

USE_KMEANS_INIT = True
USE_AMP = True
MIXED_PRECISION = "fp16"
WANDB_LOGGING = True
WANDB_PROJECT = "vmarket-rqvae-training"
WANDB_ENTITY = None
WANDB_RUN_NAME = "rqvae-128x64x32"
WANDB_SECRET_NAME = "WANDB_API_KEY"
AUTO_INSTALL_DEPENDENCIES = True
RESET_OUTPUT = False
SKIP_IF_COMPLETE = True

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys


required_modules = {
    "gin": "gin-config==0.5.0",
    "accelerate": "accelerate>=1.0.0",
    "einops": "einops>=0.8.0",
    "huggingface_hub": "huggingface-hub>=0.25.0",
    "wandb": "wandb>=0.19.0",
    "pyarrow": "pyarrow>=16.0.0",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if AUTO_INSTALL_DEPENDENCIES and missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training RQ-VAE.")

## 2. Kết nối Weights & Biases

Tạo Kaggle Secret tên `WANDB_API_KEY` và cấp quyền cho notebook. API key chỉ được đưa vào environment của kernel và subprocess, không được ghi vào Gin config hoặc output artifact.

In [ ]:
import os


if WANDB_LOGGING:
    from kaggle_secrets import UserSecretsClient

    api_key = UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    os.environ["WANDB_API_KEY"] = api_key
    import wandb

    wandb.login(key=api_key, relogin=True)
    print("Weights & Biases login: PASSED")
    print("W&B project:", WANDB_PROJECT)
else:
    print("Weights & Biases logging is disabled.")

## 3. Clone source từ GitHub

Cell này đọc token từ Kaggle Secret `GITHUB_TOKEN` và clone nhánh `main`. Token được cấp cho Git qua `GIT_ASKPASS`, không xuất hiện trong URL, command log hoặc Git config.

In [ ]:
import os


if CLONE_SOURCE_FROM_GITHUB:
    from kaggle_secrets import UserSecretsClient

    github_token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)

    if REPOSITORY_ROOT is None:
        REPOSITORY_ROOT = (
            Path("/kaggle/working/vsf-miniapp-ecommerce-source")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "vsf-miniapp-ecommerce-source"
        )
    REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
    askpass_path = REPOSITORY_ROOT.parent / ".github-askpass.sh"
    askpass_path.parent.mkdir(parents=True, exist_ok=True)
    askpass_path.write_text(
        "#!/usr/bin/env python3\n"
        "import os\n"
        "import sys\n"
        "prompt = sys.argv[1] if len(sys.argv) > 1 else ''\n"
        "if 'Username' in prompt:\n"
        "    print('x-access-token')\n"
        "elif 'Password' in prompt:\n"
        "    print(os.environ['GITHUB_TOKEN'])\n",
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    git_environment = os.environ.copy()
    git_environment.update({
        "GITHUB_TOKEN": github_token,
        "GIT_ASKPASS": str(askpass_path),
        "GIT_TERMINAL_PROMPT": "0",
    })
    try:
        if (REPOSITORY_ROOT / ".git").is_dir():
            subprocess.check_call(
                ["git", "-C", str(REPOSITORY_ROOT), "fetch", "--depth", "1", "origin", GITHUB_BRANCH],
                env=git_environment,
            )
            subprocess.check_call(
                ["git", "-C", str(REPOSITORY_ROOT), "checkout", GITHUB_BRANCH],
                env=git_environment,
            )
            subprocess.check_call(
                ["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
                env=git_environment,
            )
        elif REPOSITORY_ROOT.exists():
            raise FileExistsError(f"Clone target exists but is not a Git repository: {REPOSITORY_ROOT}")
        else:
            subprocess.check_call(
                [
                    "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                    GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT),
                ],
                env=git_environment,
            )
    finally:
        askpass_path.unlink(missing_ok=True)
        del github_token
        git_environment.pop("GITHUB_TOKEN", None)

    SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
    if not (SOURCE_ROOT / "train_rqvae.py").is_file():
        raise FileNotFoundError(f"RQ-VAE source was not found after clone: {SOURCE_ROOT}")
    print("Repository ready:", REPOSITORY_ROOT)
    print("SOURCE_ROOT:", SOURCE_ROOT)
else:
    print("GitHub source cloning is disabled.")

## 4. Tìm source và embedding artifacts

Source hiện dùng import phẳng như `from data...` và `from modules...`. Vì vậy notebook sẽ chạy trainer với `cwd` chính là thư mục `src`, thay vì import từ thư mục gốc repository.

In [ ]:
def is_source_root(path):
    path = Path(path)
    return (
        (path / "train_rqvae.py").is_file()
        and (path / "configs/rqvae_vmarket.gin").is_file()
        and (path / "modules/rqvae.py").is_file()
        and (path / "data/vmarket.py").is_file()
    )


def locate_source_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_source_root(root):
            return root
        raise FileNotFoundError(f"RQ-VAE source was not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "src",
        cwd / "ai-recommendation/src",
        cwd.parent / "ai-recommendation/src",
        Path("/kaggle/working/ai-recommendation/src"),
    ]
    for candidate in candidates:
        if is_source_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for trainer_path in sorted(kaggle_input.glob("**/train_rqvae.py")):
            if is_source_root(trainer_path.parent):
                return trainer_path.parent.resolve()
    raise FileNotFoundError(
        "RQ-VAE source was not found. Add the current src folder as Kaggle Data or set SOURCE_ROOT."
    )


def is_embedding_root(path):
    path = Path(path)
    return (
        (path / "global_product_embeddings.f16.npy").is_file()
        and (path / "global_embedding_index.parquet").is_file()
    )


def locate_embedding_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_embedding_root(root):
            return root
        raise FileNotFoundError(f"Notebook 02 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/embeddings"),
        cwd / "embeddings",
        cwd.parent / "embeddings",
    ]
    for candidate in candidates:
        if is_embedding_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for embedding_path in sorted(kaggle_input.glob("**/global_product_embeddings.f16.npy")):
            if is_embedding_root(embedding_path.parent):
                return embedding_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 02 output was not found. Add it as Kaggle Data or set EMBEDDING_ROOT."
    )


SOURCE_ROOT = locate_source_root(SOURCE_ROOT)
EMBEDDING_ROOT = locate_embedding_root(EMBEDDING_ROOT)
if OUTPUT_ROOT is None:
    OUTPUT_ROOT = (
        Path("/kaggle/working/vmarket_rqvae")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "vmarket_rqvae"
    )
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()

print("SOURCE_ROOT:", SOURCE_ROOT)
print("EMBEDDING_ROOT:", EMBEDDING_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 5. Kiểm tra input contract

In [ ]:
embedding_path = EMBEDDING_ROOT / "global_product_embeddings.f16.npy"
index_path = EMBEDDING_ROOT / "global_embedding_index.parquet"

embedding_matrix = np.load(embedding_path, mmap_mode="r")
index_file = pq.ParquetFile(index_path)

if embedding_matrix.ndim != 2 or embedding_matrix.shape[1] != 256:
    raise ValueError(f"Expected embeddings with shape [N, 256], found {embedding_matrix.shape}")
if embedding_matrix.dtype != np.float16:
    raise ValueError(f"Expected float16 embeddings, found {embedding_matrix.dtype}")
if index_file.metadata.num_rows != len(embedding_matrix):
    raise ValueError(
        f"Embedding/index row mismatch: {len(embedding_matrix)} != {index_file.metadata.num_rows}"
    )
required_index_columns = {"product_index", "product_id"}
if not required_index_columns.issubset(index_file.schema_arrow.names):
    raise ValueError(f"Missing index columns: {required_index_columns - set(index_file.schema_arrow.names)}")

print("Input validation: PASSED")
print("Products:", f"{len(embedding_matrix):,}")
print("Embedding shape:", embedding_matrix.shape)
print("Embedding size:", f"{embedding_path.stat().st_size / 2**30:.2f} GiB")
del embedding_matrix

## 6. Chuẩn bị source runtime và Gin config

Kaggle mount `/kaggle/input` ở chế độ chỉ đọc. Cell này copy source sang `/kaggle/working/vmarket_rqvae_src`, sau đó chỉ sửa bản copy runtime.

In [ ]:
import json
import re
import shutil


RUNTIME_SOURCE_ROOT = (
    Path("/kaggle/working/vmarket_rqvae_src")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "vmarket_rqvae_src"
).resolve()
if SOURCE_ROOT != RUNTIME_SOURCE_ROOT:
    shutil.copytree(SOURCE_ROOT, RUNTIME_SOURCE_ROOT, dirs_exist_ok=True)

if RESET_OUTPUT and OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.name != "vmarket_rqvae":
        raise ValueError(f"Refusing to reset an unexpected output path: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

base_config_path = RUNTIME_SOURCE_ROOT / "configs/rqvae_vmarket.gin"
runtime_config_path = RUNTIME_SOURCE_ROOT / "configs/rqvae_kaggle_runtime.gin"
config_text = base_config_path.read_text(encoding="utf-8")

overrides = {
    "train.iterations": str(ITERATIONS),
    "train.batch_size": str(BATCH_SIZE),
    "train.learning_rate": repr(LEARNING_RATE),
    "train.weight_decay": repr(WEIGHT_DECAY),
    "train.dataset_folder": json.dumps(str(EMBEDDING_ROOT)),
    "train.save_dir_root": json.dumps(str(OUTPUT_ROOT)),
    "train.vae_codebook_sizes": json.dumps(list(CODEBOOK_SIZES)),
    "train.eval_fraction": repr(EVAL_FRACTION),
    "train.split_seed": str(SEED),
    "train.eval_every": str(EVAL_EVERY),
    "train.save_model_every": str(SAVE_MODEL_EVERY),
    "train.use_kmeans_init": str(USE_KMEANS_INIT),
    "train.amp": str(USE_AMP),
    "train.mixed_precision_type": json.dumps(MIXED_PRECISION),
    "train.wandb_logging": str(WANDB_LOGGING),
    "train.wandb_project": json.dumps(WANDB_PROJECT),
    "train.wandb_entity": "None" if WANDB_ENTITY is None else json.dumps(WANDB_ENTITY),
    "train.wandb_run_name": "None" if WANDB_RUN_NAME is None else json.dumps(WANDB_RUN_NAME),
}
for key, value in overrides.items():
    pattern = rf"(?m)^{re.escape(key)}=.*$"
    config_text, replacement_count = re.subn(pattern, f"{key}={value}", config_text)
    if replacement_count != 1:
        raise ValueError(f"Expected exactly one Gin binding for {key}, found {replacement_count}")

if PRETRAINED_CHECKPOINT is not None:
    checkpoint_path = Path(PRETRAINED_CHECKPOINT).expanduser().resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    config_text += f"\ntrain.pretrained_rqvae_path={json.dumps(str(checkpoint_path))}\n"

runtime_config_path.write_text(config_text, encoding="utf-8")

print("RUNTIME_SOURCE_ROOT:", RUNTIME_SOURCE_ROOT)
print("Runtime Gin config:", runtime_config_path)
print("Codebook sizes:", CODEBOOK_SIZES)
print("Theoretical SID space:", int(np.prod(CODEBOOK_SIZES)))
print("\n" + config_text)

## 7. Train RQ-VAE

Cell này stream log trực tiếp từ subprocess. Trainer lưu checkpoint định kỳ vào `OUTPUT_ROOT`. Nếu Kaggle session dừng, add output checkpoint của session trước làm Data và đặt `PRETRAINED_CHECKPOINT` khi chạy lại.

In [ ]:
import os


semantic_ids_path = OUTPUT_ROOT / "semantic_ids.parquet"
metrics_path = OUTPUT_ROOT / "semantic_id_metrics.json"
training_complete = semantic_ids_path.is_file() and metrics_path.is_file()

if SKIP_IF_COMPLETE and training_complete:
    print("Complete RQ-VAE artifacts already exist; skipping training.")
else:
    command = [sys.executable, "train_rqvae.py", str(runtime_config_path)]
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    environment["WANDB_SILENT"] = "true"

    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=RUNTIME_SOURCE_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"RQ-VAE training failed with exit code {return_code}")

print("Training cell finished.")

## 8. Kiểm tra artifacts và cluster SID

In [ ]:
if not semantic_ids_path.is_file() or not metrics_path.is_file():
    raise FileNotFoundError("RQ-VAE finished without the required final artifacts.")

semantic_file = pq.ParquetFile(semantic_ids_path)
expected_columns = ["product_index", "product_id", "sid_0", "sid_1", "sid_2"]
if semantic_file.schema_arrow.names != expected_columns:
    raise ValueError(f"Unexpected semantic ID schema: {semantic_file.schema_arrow.names}")

sid_table = pq.read_table(semantic_ids_path, columns=["sid_0", "sid_1", "sid_2"])
sid_ranges = {}
for layer, codebook_size in enumerate(CODEBOOK_SIZES):
    values = sid_table.column(f"sid_{layer}").combine_chunks().to_numpy()
    minimum = int(values.min())
    maximum = int(values.max())
    if minimum < 0 or maximum >= codebook_size:
        raise ValueError(
            f"sid_{layer} outside [0, {codebook_size - 1}]: {minimum}..{maximum}"
        )
    sid_ranges[f"sid_{layer}"] = {"min": minimum, "max": maximum}

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
checkpoints = sorted(OUTPUT_ROOT.glob("checkpoint_*.pt"))
output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Final artifact validation: PASSED")
print("Semantic ID rows:", f"{semantic_file.metadata.num_rows:,}")
print("SID ranges:", json.dumps(sid_ranges, indent=2))
print("Checkpoints:", len(checkpoints))
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("OUTPUT_ROOT:", OUTPUT_ROOT)
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

## Khi nào notebook hoàn thành?

Notebook hoàn thành khi cell cuối báo `Final artifact validation: PASSED`. Sau đó lưu một Kaggle Notebook Version để giữ toàn bộ thư mục `/kaggle/working/vmarket_rqvae` làm output cho bước phân tích cluster và huấn luyện Transformer tiếp theo.